In [1]:
import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device("cpu") #bc i'm doing this in aws, no gpu needed
transform = transforms.ToTensor() #this is a torchvision package for casting images to torch.Tensor
test_ds = datasets.MNIST(root="", train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=56, shuffle=False)
model = nn.Sequential(
    nn.Flatten(), #unsurprisingly, this flattens data 0
    nn.Linear(28*28, 256), #linear layer 1 z1
    nn.ReLU(), # 2 h1
    nn.Linear(256, 128), #squash down 3 z2
    nn.ReLU(), # 4 h2
    nn.Linear(128, 10) #output 5 logits
).to(device)
#each linear layer is an n x n matrix. the first layer is 28*28 whatever values multiplied down to 256 values
model.load_state_dict(torch.load("mlp_mnist_state_dict.pt", map_location=device))

def get_activations_for_all_digits(model):
    acts = {}
    digit_activations = {x: [] for x in range(10)}

    def save_activation(name):
        def hook(module, inp, out):
            acts[name] = out.detach()
        return hook

    h1_monitor = model[2].register_forward_hook(save_activation("h1"))#create forward hook on this and call it a1
    z2_monitor = model[3].register_forward_hook(save_activation("z2"))
    logits_monitor = model[5].register_forward_hook(save_activation("logits"))

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)

            _ = model(x)
            
            h1 = acts.pop("h1")#these are of length(batch) and contain activations for each value in y 
            z2 = acts.pop("z2")
            logits = acts.pop("logits")


            for i in range(len(y)):#iterate through y
                digit = y[i].item()#cast tensor to python int
                digit_activations[digit].append([h1[i], z2[i], logits[i]])#take slices out of a1, a2, and a3

        h1_monitor.remove()#remove hooks when done
        z2_monitor.remove()
        logits_monitor.remove()

        return digit_activations
    

In [2]:
results = get_activations_for_all_digits(model)

In [22]:
avg_activations = {}
for digit in range(10):
    temp_layer_2_avg = torch.stack([act[1] for act in results[digit]]).mean(dim=0)
    avg_activations[digit] = temp_layer_2_avg

In [23]:
loopy_digits = (0, 6, 8, 9)

In [28]:
loopy_sum = torch.stack([avg_activations[d] for d in loopy_digits]).sum(dim=0)
non_loopy_sum = torch.stack([avg_activations[d] for d in range(10) if d not in loopy_digits]).sum(dim=0)

In [30]:
loopy_dif = (loopy_sum * 1.25) - non_loopy_sum

In [32]:
print(loopy_dif.shape)

torch.Size([128])
